# 01 — Data Importing and Overview
### Introduction:

This notebook imports the raw exploration datasets from the 2023 MPD drill program and performs an initial inspection before any cleaning or transformation begins.


### Objectives:
- Load raw drillhole, assay, and collar datasets
- Standardize column names across tables
- Verify dataset integrity and hole ID alignment
- Export raw tables to Data/raw for cleaning in Notebook 02

### Contents:
**1. Setup and Libraries**

**2. Load Raw CSV Files**

**3. Add Collar Table**

**4. Inspect Raw Table Structure**

**5. Standardize Column Names**

**6. Align Key Drill Log Columns**

**7. Preview Standardized Raw Tables**

**8. Missing Values and Hole ID Check**

**9. Compare Hole IDs Across Drill and Assay Datasets**

**10. Export Standardized Tables**

**11. Conclusions**


### 1. Setup and Libraries:

The first step is to import the required libraries and define the path to the raw data folder.


In [2]:
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# The notebook folder is /Notebooks, so the project root is the parent folder
raw_data = Path('../Data/raw')

raw_data


PosixPath('../Data/raw')

### 2. Load Raw CSV Files:

Three raw tables are loaded for this analysis — drill logs, drill assays, and collar coordinates. The drill log and assay files are read directly from the raw data folder. Collar coordinates were not available in the CSV appendices and are added manually in Section 3.

In [5]:
# Function to load CSV files, removing unnamed columns
def load_csv(filename, header_row=0):
    # Read the CSV file into a DataFrame, specifying the header row
    df = pd.read_csv(raw_data / filename, header=header_row)
    # Remove unnamed columns - pandas creates unnamed column headers for empty columns, which can occur due to formatting issues in the CSV files
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
    return df
# Load the drill and assay data using the load_csv function, specifying the appropriate header rows for each file
drill_df = load_csv("Appendix A_2023 Drill Data.csv", header_row=1)
assay_df = load_csv("Appendix B_2023 DDH Assays.csv", header_row=0)


In [6]:
print(f"drill_df: {drill_df.shape[0]} rows x {drill_df.shape[1]} columns")
print(f"assay_df: {assay_df.shape[0]} rows x {assay_df.shape[1]} columns")

drill_df: 1430 rows x 21 columns
assay_df: 6635 rows x 116 columns


### 3. Add Collar Table:

Collar coordinates are not available in the same CSV files, so this table was manually transcribed from the ARIS assessment PDF report. Collar data is required for later spatial validation and integration.


In [7]:
collar_data = [
    ["AXE-23-001", 677400, 5503115, 1418, 180, -65, 732, "West", "Completed"],
    ["AXE-23-002", 677400, 5503115, 1418,   0, -90, 819.01, "West", "Completed"],
    ["AXE-23-003", 677400, 5503115, 1418,  90, -45, 367, "West", "Completed"],
    ["AXE-23-004", 677400, 5503115, 1418,  90, -75, 707, "West", "Completed"],
    ["AXE-23-005", 677400, 5503115, 1418,  25, -50, 87.35, "West", "Abandoned"],
    ["AXE-23-006", 677400, 5503120, 1418,  15, -45, 97.35, "West", "Abandoned"],
    ["AXE-23-007", 677400, 5503120, 1418,  15, -50, 459, "West", "Completed"],
    ["AXE-23-008", 677397, 5502825, 1398, 350, -75, 897, "West", "Completed"],
    ["AXE-23-009", 677397, 5502825, 1398,  90, -85, 83, "West", "Abandoned"],
    ["AXE-23-010", 677397, 5502825, 1398,  90, -80, 709.1, "West", "Completed"],
    ["AXE-23-011", 677397, 5502930, 1398,   0, -90, 1031, "West", "Completed"],
    ["AXE-23-012", 678515, 5501650, 1310, 100, -68, 849, "South", "Completed"],
    ["AXE-23-013", 678515, 5501650, 1335, 305, -80, 944, "South", "Completed"],
    ["AXE-23-014", 678515, 5501650, 1310, 345, -57, 1061.2, "South", "Completed"],
    ["AXE-23-015", 680135, 5501665, 1000,   0, -90, 253, "1516", "Ended"],
    ["AXE-23-016", 680135, 5501665, 1000,  90, -80, 782, "1516", "Completed"],
    ["AXE-23-017", 680135, 5501665, 1000,  90, -50, 600, "1516", "Completed"],
    ["AXE-23-018", 680135, 5501665, 1000, 20.17, -51, 938, "1516", "Completed"],
    ["AXE-23-019", 680139, 5501677,  995, 145.35, -50, 146, "1516", "Ended"],
    ["AXE-23-020", 680139, 5501677,  995, 145, -60, 129, "1516", "Ended"],
    ["AXE-23-021", 680344, 5501891, 1150, 290, -75, 183, "1516", "Ended"],
    ["MPD-23-001", 681435, 5513816, 1360,   0, -90, 995, "Man", "Completed"],
    ["MPD-23-002", 681435, 5513816, 1360,  90, -70, 924, "Man", "Completed"],
    ["MPD-23-003", 681435, 5513816, 1360,  90, -80, 1094, "Man", "Completed"],
    ["MPD-23-004", 681434, 5513814, 1360, 272, -60, 104, "Man*", "Abandoned"],
    ["MPD-23-005", 681434, 5513814, 1360, 272, -65, 825, "Man*", "Completed"],
    ["MPD-23-006", 681435, 5513816, 1360, 272, -80, 879, "Man", "Completed"],
    ["MPD-23-007", 681435, 5513816, 1360, 342.6, -50, 807, "Man", "Completed"],
    ["MPD-23-008", 681360, 5513440, 1364,  90, -45, 54, "Beyer", "Abandoned"],
    ["MPD-23-009", 681360, 5513440, 1364,  90, -60, 487, "Beyer", "Completed"],
    ["MPD-23-010", 681416, 5513389, 1387, 343, -65, 295, "Beyer", "Completed"],
    ["MPD-23-011", 681416, 5513389, 1387, 333, -68, 80, "Beyer", "Completed"],
    ["MPD-23-012", 681416, 5513389, 1387, 333, -50, 144, "Beyer", "Completed"],
]

collar_df = pd.DataFrame(collar_data, columns=[
    "hole_id", "easting", "northing", "elevation_m",
    "azimuth_deg", "dip_deg", "length_m", "area", "status"
])

collar_df.head()


,hole_id,easting,northing,elevation_m,azimuth_deg,dip_deg,length_m,area,status
0,AXE-23-001,677400,5503115,1418,180.0,-65,732.00,West,Completed
1,AXE-23-002,677400,5503115,1418,0.0,-90,819.01,West,Completed
2,AXE-23-003,677400,5503115,1418,90.0,-45,367.00,West,Completed
3,AXE-23-004,677400,5503115,1418,90.0,-75,707.00,West,Completed
4,AXE-23-005,677400,5503115,1418,25.0,-50,87.35,West,Abandoned


### 4. Inspect Raw Table Structure:

We want to confirm the initial shape and column names before performing any structural standardization.


In [8]:
print("drill_df shape:", drill_df.shape)
print("assay_df shape:", assay_df.shape)
print("collar_df shape:", collar_df.shape)

print("Drill columns:")
print(drill_df.columns.tolist())
print("Assay columns:")
print(assay_df.columns.tolist())


drill_df shape: (1430, 21)
assay_df shape: (6635, 116)
collar_df shape: (33, 9)
Drill columns:
['Hole \nNumber', 'From \n(m)', 'To \n(m)', 'Base \nLithology', 'Litho \nDetailed', 'Lith \nGeneral', 'Lith\nGrain Size', 'Lith\nTexture1', 'Lith \nTexture2', 'Lith\nTexture3', 'Lith Description', 'Alt \nFrom (m)', 'Alt\nTo (m)', 'Dominant \nAlteration', 'Intensity', 'Dominant Alt Mins', 'Min \nFrom (m)', 'Min\nTo (m)', 'Py pct\nNV*', 'Ccp pct\nNV*', 'Bn pct\nNV*']
Assay columns:
['Hole number', 'From', 'To', 'Sample', 'Sample Type', 'Standard Type', 'Parent Sample Number', 'Sample Size', 'Sampled By', 'Comments', 'Certificate', 'Certificate completed', 'Ag ppm Ag-OG62', 'Ag ppm ME-MS61', 'Al % ME-MS61', 'As ppm ME-MS61', 'Au ppm Au-AA24', 'Au ppm \nAu-GRA22', 'Ba ppm ME-MS61', 'Be ppm ME-MS61', 'Bi ppm ME-MS61', 'Ca % ME-MS61', 'Cd ppm ME-MS61', 'Ce ppm ME-MS61', 'Co ppm ME-MS61', 'Cr ppm ME-MS61', 'Cs ppm ME-MS61', 'Cu % \nCu-OG62', 'Cu ppm ME-MS61', 'Digest Run NA Ag-OG62', 'Digest Run NA 

### 5. Standardize Column Names:

The drill and assay tables use different column formatting conventions. We standardize names now to make later comparisons easier.


In [6]:
def clean_columns(df):
    # Work on a copy so the original DataFrame is not modified
    df = df.copy()
    df.columns = (
        df.columns
        # Replace newline characters with a space — common in Excel exported headers
        .str.replace("\n", " ", regex=False)
        # Remove leading and trailing whitespace
        .str.strip()
        # Replace any whitespace with an underscore
        .str.replace(r"\s+", "_", regex=True)
        # Remove any character that is not a letter, number, or underscore
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
        # Convert to lowercase
        .str.lower()
    )
    return df

drill_df = clean_columns(drill_df)
assay_df = clean_columns(assay_df)

print("Standardized drill columns:")
print(drill_df.columns.tolist())
print("\nStandardized assay columns:")
print(assay_df.columns.tolist())

Standardized drill columns:
['hole_number', 'from_m', 'to_m', 'base_lithology', 'litho_detailed', 'lith_general', 'lith_grain_size', 'lith_texture1', 'lith_texture2', 'lith_texture3', 'lith_description', 'alt_from_m', 'alt_to_m', 'dominant_alteration', 'intensity', 'dominant_alt_mins', 'min_from_m', 'min_to_m', 'py_pct_nv', 'ccp_pct_nv', 'bn_pct_nv']

Standardized assay columns:
['hole_number', 'from', 'to', 'sample', 'sample_type', 'standard_type', 'parent_sample_number', 'sample_size', 'sampled_by', 'comments', 'certificate', 'certificate_completed', 'ag_ppm_agog62', 'ag_ppm_mems61', 'al__mems61', 'as_ppm_mems61', 'au_ppm_auaa24', 'au_ppm_augra22', 'ba_ppm_mems61', 'be_ppm_mems61', 'bi_ppm_mems61', 'ca__mems61', 'cd_ppm_mems61', 'ce_ppm_mems61', 'co_ppm_mems61', 'cr_ppm_mems61', 'cs_ppm_mems61', 'cu__cuog62', 'cu_ppm_mems61', 'digest_run_na_agog62', 'digest_run_na_cuog62', 'digest_run_na_meicp61i', 'digest_run_na_mems61i', 'digest_run_na_znog62', 'digest_seq_na_agog62', 'digest_seq_n

### 6. Align Key Drill Log Columns:

The drill log uses `from_m` and `to_m`, while the assay table uses `from` and `to`. We rename the drill log columns so the two tables align more naturally for later merging.


In [7]:
drill_df = drill_df.rename(columns={"from_m": "from", "to_m": "to"})
print("drill_df columns after renaming:")
print(drill_df.columns.tolist())


drill_df columns after renaming:
['hole_number', 'from', 'to', 'base_lithology', 'litho_detailed', 'lith_general', 'lith_grain_size', 'lith_texture1', 'lith_texture2', 'lith_texture3', 'lith_description', 'alt_from_m', 'alt_to_m', 'dominant_alteration', 'intensity', 'dominant_alt_mins', 'min_from_m', 'min_to_m', 'py_pct_nv', 'ccp_pct_nv', 'bn_pct_nv']


### 7. Preview Standardized Raw Tables:

A quick preview to confirm column names are standardized and the 
data loaded correctly before exporting.

In [8]:
display(drill_df.head(5))
display(assay_df.head(5))


,hole_number,from,to,base_lithology,litho_detailed,lith_general,lith_grain_size,lith_texture1,lith_texture2,lith_texture3,...,alt_from_m,alt_to_m,dominant_alteration,intensity,dominant_alt_mins,min_from_m,min_to_m,py_pct_nv,ccp_pct_nv,bn_pct_nv
0,AXE-23-001,0.0,2.00,OVB,OVB,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AXE-23-001,2.0,6.00,CL,CL,NC,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AXE-23-001,6.0,17.00,DIO,DIO,SVI,FMG,XEN,SER,POR,...,6.0,12.20,PRO,4.0,Chl+act+ep,6.0,17.00,2,0.1,0.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.2,17.00,PHY,4.0,Ser+py,NaN,NaN,NaN,NaN,NaN
4,AXE-23-001,17.0,21.07,DIO,DIO,VCB,FMG,POR,XEN,NaN,...,17.0,21.07,PHY,4.0,Ser+py+/-qtz,17.0,21.07,0.25,0.0,0.0


,hole_number,from,to,sample,sample_type,standard_type,parent_sample_number,sample_size,sampled_by,comments,...,wt_total_g_pulqc,y_ppm_mems61,zn__znog62,zn_ppm_mems61,zr_ppm_mems61,cu_best_,au_best_ppm,ag_best_ppm,mo_best_ppm,zn_best_ppm
0,AXE-23-001,6.0,8.0,137255,REG,NaN,NaN,HC,Unknown,NaN,...,NaN,13.4,NaN,86,27.8,0.4660,0.174,1.18,3.98,86
1,AXE-23-001,8.0,11.0,137256,REG,NaN,NaN,HC,Unknown,NaN,...,NaN,13.4,NaN,236,29.2,0.2460,0.159,0.95,3.12,236
2,AXE-23-001,11.0,14.0,137257,REG,NaN,NaN,HC,Unknown,NaN,...,NaN,10.2,NaN,64,42.6,0.1070,0.196,0.38,1.33,64
3,AXE-23-001,14.0,17.0,137258,REG,NaN,NaN,HC,Unknown,NaN,...,NaN,13.8,NaN,57,28.5,0.3110,0.103,1.01,0.96,57
4,AXE-23-001,17.0,20.0,137259,REG,NaN,NaN,HC,Unknown,NaN,...,NaN,18.9,NaN,49,39.6,0.1215,0.104,0.35,1.53,49


### 8. Missing Values and Hole ID Check:

Check for completely empty rows (no data) and rows with missing hole id's ('hole_number') in the drill and assay table

In [9]:
print("Drill records with all values missing:", drill_df.isna().all(axis=1).sum())
print("Assay records with all values missing:", assay_df.isna().all(axis=1).sum())

missing_drill_holes = drill_df[drill_df["hole_number"].isna()]
missing_assay_holes = assay_df[assay_df["hole_number"].isna()]

print("Drill rows with missing hole_number:", missing_drill_holes.shape[0])
print("Assay rows with missing hole_number:", missing_assay_holes.shape[0])


Drill records with all values missing: 0
Assay records with all values missing: 0
Drill rows with missing hole_number: 526
Assay rows with missing hole_number: 0


### 9. Compare Hole IDs Across Drill and Assay Datasets:

We want to understand whether any holes exist only in the drill log or only in the assay table.


In [ ]:
# Create sets of unique hole numbers from both dataframes, dropping any NaN values
drill_holes = set(drill_df["hole_number"].dropna().unique())
assay_holes = set(assay_df["hole_number"].dropna().unique())
# Compare the sets to find holes that are in one dataframe but not the other
print("Holes in drill log but not in assay table:", sorted(drill_holes - assay_holes))
print("Holes in assay table but not in drill log:", sorted(assay_holes - drill_holes))


Holes in drill log but not in assay table: ['AXE-23-020', 'MPD-23-004', 'MPD-23-011']
Holes in assay table but not in drill log: []


### 10. Export Raw Tables:

The drill log containing geological interpretations, assay table containing assay results, and collar table containing drillhole coordinates are exported 
to Data/raw with column names standardized and unnamed columns removed. No further cleaning is applied here — Notebook 02 handles data structure and geological interval integrity.

In [11]:
drill_df.to_csv(raw_data / "drill_raw.csv", index=False)
assay_df.to_csv(raw_data / "assay_raw.csv", index=False)
collar_df.to_csv(raw_data / "collar_raw.csv", index=False)
print("Saved drill_raw.csv, assay_raw.csv, collar_raw.csv")


Saved drill_raw.csv, assay_raw.csv, collar_raw.csv


### 11. Conclusions:

- Drill log and assay CSV files were downloaded directly from the BC ARIS portal for Assessment Report 42616
- Collar coordinates were not available in the CSV appendices and were manually transcribed from the ARIS PDF report — covering all drillholes in the 2023 program with easting, northing, elevation, azimuth, dip, and total depth
- Column names were standardized across the drill and assay tables using a consistent lowercase underscore convention, making later merging straightforward
- 526 drill rows were identified with missing hole identifiers — a result of Excel merged cells in the original source file
- All holes present in the drill log were confirmed in the assay table and vice versa — no holes are missing from either dataset
- All three tables are exported to the project's Data/raw folder with column names standardized and unnamed columns removed